# Tutorial: Time Series Analysis

> **Type:** Tutorial — follow along step by step. Your only job is to make it work. Don't skip cells.
>
> **Time:** ~1.5-2 hours
> **Prerequisites:** Module 09 (EDA), Module 13 (cross-validation), Module 16 (model evaluation)
> **What you'll build:** A full forecasting pipeline on a synthetic seasonal demand series — decomposition, a stationarity test, ACF/PACF-driven order selection, a fitted SARIMA model, a forecast with confidence intervals, and a proper time-series evaluation.

---

## What We're Building

We'll generate a realistic daily demand series with trend, weekly seasonality, and noise (so the exercise is reproducible without needing internet access to fetch a dataset). Then we'll: visually inspect it, decompose it into trend/seasonal/residual, test and achieve stationarity, use ACF/PACF to choose a SARIMA order, fit the model, forecast the next 28 days with a confidence interval, and evaluate against a proper chronological holdout using MAE/RMSE/MAPE.

Don't worry yet about understanding every line — that's what `explanation.md` is for. Here, just follow the steps and watch it work.

## Step 0: Setup

Install and import everything we need.

In [ ]:
# !pip install numpy pandas matplotlib statsmodels scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
print('Setup complete.')

## Step 1: Generate the Data

We'll simulate two years of daily "units sold" data for a fictional product: a gently rising trend, a strong weekly seasonal pattern (higher on weekends), and additive Gaussian noise. This mirrors the shape of real retail demand data closely enough to practise on, without needing a download.

In [ ]:
n_days = 730  # 2 years
dates = pd.date_range('2023-01-01', periods=n_days, freq='D')

trend = np.linspace(100, 180, n_days)                          # slow linear growth
day_of_week = dates.dayofweek.values                             # 0=Mon ... 6=Sun
weekly_seasonal = np.where(day_of_week >= 5, 25, 0) + np.where(day_of_week == 4, 10, 0)  # weekend bump
noise = np.random.normal(0, 8, n_days)

demand = trend + weekly_seasonal + noise
demand = np.maximum(demand, 0)  # units sold can't be negative

df = pd.DataFrame({'date': dates, 'demand': demand}).set_index('date')
df.head()

> **Check:** You should see a DataFrame indexed by date with a single `demand` column, 730 rows. Values should be roughly in the 90-220 range.

## Step 2: Visualise Trend and Seasonality

Always look at the raw series before doing anything else.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['demand'], linewidth=1, color='#4E79A7')
ax.set_title('Daily Demand — Raw Series')
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold')
plt.tight_layout()
plt.show()

# Zoom into 6 weeks to see the weekly pattern clearly
fig, ax = plt.subplots(figsize=(14, 4))
subset = df.iloc[:42]
ax.plot(subset.index, subset['demand'], marker='o', color='#E15759')
ax.set_title('First 6 Weeks — Weekly Seasonality Visible')
plt.tight_layout()
plt.show()

> **What just happened?** The full series shows a clear upward trend. The 6-week zoom shows a repeating weekly bump — exactly the two components we simulated. In a real dataset, this visual pass is how you'd first notice these patterns exist.

## Step 3: Seasonal Decomposition

Split the series into trend, seasonal, and residual components explicitly.

In [ ]:
decomposition = seasonal_decompose(df['demand'], model='additive', period=7)

fig = decomposition.plot()
fig.set_size_inches(13, 8)
plt.tight_layout()
plt.show()

print(f"Residual std: {decomposition.resid.std():.2f}")
print(f"(Compare to the noise std we simulated: 8.0 — decomposition recovered it closely)")

> **What just happened?** `seasonal_decompose` fit a trend (via moving average), extracted a repeating 7-day seasonal pattern, and left the residual. The recovered residual std should be close to the 8.0 we injected — confirming the decomposition correctly separated signal from noise.

## Step 4: Test for Stationarity

Run the Augmented Dickey-Fuller test on the raw series, then on the differenced series.

In [ ]:
def run_adf(series, label):
    result = adfuller(series.dropna())
    print(f"{label}")
    print(f"  ADF statistic: {result[0]:.4f}")
    print(f"  p-value:       {result[1]:.4f}")
    print(f"  Stationary?    {'Yes' if result[1] < 0.05 else 'No'} (threshold p < 0.05)")
    print()

run_adf(df['demand'], 'Raw series')

df['demand_diff1'] = df['demand'].diff()
run_adf(df['demand_diff1'], 'First-differenced series')

> **What just happened?** The raw series has a p-value above 0.05 — we can't reject the null of non-stationarity (unsurprising, it has a trend). After first-differencing (`yₜ - yₜ₋₁`), the p-value drops well below 0.05 — the differenced series is stationary. This tells us d=1 in our eventual ARIMA/SARIMA order.

## Step 5: ACF and PACF — Choosing Model Order

Plot the autocorrelation and partial autocorrelation of the differenced series to pick p and q (and the seasonal terms).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df['demand_diff1'].dropna(), lags=21, ax=axes[0])
axes[0].set_title('ACF — Differenced Series')
plot_pacf(df['demand_diff1'].dropna(), lags=21, ax=axes[1])
axes[1].set_title('PACF — Differenced Series')
plt.tight_layout()
plt.show()

> **What just happened?** Look for a spike at lag 7 (and multiples of 7) in the ACF — that's the fingerprint of weekly seasonality, confirming what we saw in the decomposition. A sharp PACF cutoff after a low lag suggests a small AR order; a slowly-decaying-then-spiking-at-7 ACF is exactly the signature that tells us to add a **seasonal** term at m=7 rather than trying to capture it with a huge non-seasonal AR order.

## Step 6: Chronological Train/Test Split

Never shuffle time series data. Hold out the last 28 days as the test set; train on everything before that.

In [ ]:
horizon = 28
train = df['demand'].iloc[:-horizon]
test = df['demand'].iloc[-horizon:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} days)")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} days)")

## Step 7: Fit a SARIMA Model

Based on Steps 4-5: d=1 (one round of differencing for stationarity), weekly seasonality at m=7. We'll use a modest order — SARIMA(1,1,1)(1,1,1,7) — a reasonable starting point given a single, clear weekly cycle.

In [ ]:
model = SARIMAX(
    train,
    order=(1, 1, 1),            # (p, d, q) — non-seasonal
    seasonal_order=(1, 1, 1, 7), # (P, D, Q, m) — seasonal, weekly period
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fitted = model.fit(disp=False)
print(fitted.summary())

> **What just happened?** `SARIMAX` fit the AR, MA, seasonal AR, and seasonal MA coefficients by maximum likelihood. The summary's AIC/BIC let you compare this order against alternatives (try a couple of other (p,q) combinations yourself and see if AIC improves).

## Step 8: Forecast with a Confidence Interval

In [ ]:
forecast_result = fitted.get_forecast(steps=horizon)
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.05)  # 95% interval

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-60:], train.values[-60:], label='Train (last 60 days)', color='#4E79A7')
ax.plot(test.index, test.values, label='Actual (test)', color='#59A14F', linewidth=2)
ax.plot(forecast_mean.index, forecast_mean.values, label='Forecast', color='#E15759', linewidth=2, linestyle='--')
ax.fill_between(forecast_mean.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                color='#E15759', alpha=0.2, label='95% confidence interval')
ax.set_title('SARIMA Forecast vs. Actual')
ax.legend()
plt.tight_layout()
plt.show()

> **What just happened?** The forecast (dashed red) should track the actual holdout (green) reasonably closely, including the weekly bumps, with the shaded band showing the model's own uncertainty — which should widen the further out you forecast.

## Step 9: Evaluate — MAE, RMSE, MAPE

The payoff: quantify forecast accuracy on the untouched holdout.

In [ ]:
mae = mean_absolute_error(test, forecast_mean)
rmse = np.sqrt(mean_squared_error(test, forecast_mean))
mape = np.mean(np.abs((test.values - forecast_mean.values) / test.values)) * 100

print(f"MAE:  {mae:.2f} units")
print(f"RMSE: {rmse:.2f} units")
print(f"MAPE: {mape:.2f}%")

# Baseline comparison: naive forecast (repeat last observed value)
naive_forecast = np.full(horizon, train.iloc[-1])
naive_mae = mean_absolute_error(test, naive_forecast)
print(f"\nNaive baseline MAE (repeat last value): {naive_mae:.2f} units")
print(f"SARIMA improvement over naive: {(1 - mae/naive_mae) * 100:.1f}%")

> **What just happened?** We compared the SARIMA forecast against a naive "repeat the last value" baseline — the minimum bar any real forecasting model must clear. Always report this baseline alongside your model's metrics; a fancy model that barely beats it isn't earning its complexity.

## What You Built

A complete forecasting pipeline: decomposition to understand the series, an ADF test and differencing to achieve stationarity, ACF/PACF plots to choose a SARIMA order, a fitted SARIMA model, a forecast with a calibrated confidence interval, and an honest evaluation against both a chronological holdout and a naive baseline.

## What's Next

- Read `explanation.md` if you haven't already, to understand *why* stationarity and differencing matter.
- Open `howto.md` when you need to forecast your own series, choose orders faster with `auto_arima`, or build a lag-feature ML forecaster instead.
- Keep `reference.md` open as a cheat sheet for ACF/PACF interpretation and evaluation metrics.
- **Extension challenge:** Refit with a couple of different `(p,d,q)(P,D,Q,m)` orders and compare AIC. Then try building a gradient-boosted-tree forecaster using lag features (see `howto.md`) and compare its MAE against SARIMA's.

---

**AWS SageMaker version:** See the "AWS SageMaker Equivalent" section of `howto.md` for the Amazon Forecast / SageMaker DeepAR path to the same result.